# 🧪 모델 & 하드웨어 선정 평가

> **목적**: V2+ 자체 서빙을 위한 EXAONE AWQ 모델 벤치마크 테스트
> 
> **참고 문서**: `04_채팅_모델_선정.md`, `05_꼬리질문_모델_선정.md`

---

## 1. 평가 개요

### 평가 목적
- V2+ 자체 서빙을 위한 **EXAONE AWQ 모델** 최적 크기 결정 (7.8B vs 32B)
- 꼬리질문 생성 능력 및 맥락 파악 정확도 비교
- GPU 하드웨어 스펙 및 비용 최적화

### 평가 환경
| 항목 | 내용 |
|------|------|
| 런타임 | Google Colab |
| 7.8B GPU | NVIDIA L4 (24GB) |
| 32B GPU | NVIDIA A100 40GB |
| 추론 | Transformers + BitsAndBytes 4bit |

## 2. 후보 모델

### 모델 리스트
| 모델 | 파라미터 | HuggingFace ID |
|------|----------|----------------|
| **EXAONE-3.5-7.8B-AWQ** | 7.8B | `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct` |
| **EXAONE-3.5-32B-AWQ** | 32B | `LGAI-EXAONE/EXAONE-3.5-32B-Instruct` |

### 모델 크기별 특징
| 특징 | 7.8B | 32B |
|------|------|-----|
| 필요 VRAM | ~8GB (4bit) | ~20GB (4bit) |
| 추론 속도 | 빠름 (~35 TPS) | 느림 (~20 TPS) |
| 꼬리질문 품질 | 75-80점 | **85-90점** ✅ |

## 3. 하드웨어 스펙 비교

### GPU 옵션
| GPU | VRAM | 시간당 비용 | 적합 모델 |
|-----|------|------------|----------|
| NVIDIA T4 | 16GB | ~$0.50 | 7.8B (제한적) |
| NVIDIA L4 | 24GB | ~$0.80 | 7.8B ✅ |
| NVIDIA A10G | 24GB | ~$1.00 | 7.8B ✅ |
| NVIDIA A100 40GB | 40GB | ~$3.00 | 32B ✅ |

### 모델별 필요 VRAM
```
EXAONE 7.8B:  ~8GB + KV Cache ~4GB = ~12GB 권장
EXAONE 32B:   ~20GB + KV Cache ~8GB = ~28GB+ 권장
```

## 4. 성능 평가 지표

| 지표 | 설명 | 목표 |
|------|------|------|
| **TTFT** | Time To First Token | < 500ms |
| **TPS** | Tokens Per Second | > 30 tokens/sec |
| **꼬리질문 품질** | 관련성, 자연스러움 | 점수 80+ |
| **JSON 준수율** | 형식 정확도 | 100% |

---
# 🔧 환경 설정

In [ ]:
# 패키지 설치
!pip install -q transformers accelerate bitsandbytes
!pip install -q google-generativeai
!pip install -q huggingface_hub
!pip install -q torch

In [ ]:
import json
import time
import re
from typing import Dict, Any, Optional, List
from dataclasses import dataclass, field

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# Hugging Face 로그인
import os
from getpass import getpass

hf_token = getpass("🔑 Hugging Face Access Token: ")

if hf_token:
    try:
        login(token=hf_token)
        print("✅ Hugging Face 로그인 성공!")
        os.environ["HF_TOKEN"] = hf_token
    except Exception as e:
        print(f"❌ 로그인 실패: {e}")
else:
    print("⚠️ 토큰이 입력되지 않았습니다.")

In [ ]:
# Gemini API 설정 (비교용/Fallback)
GOOGLE_API_KEY = getpass("🔑 Google API Key (선택사항, 비어두면 스킵): ")

GEMINI_AVAILABLE = bool(GOOGLE_API_KEY)
gemini_client = None

if GEMINI_AVAILABLE:
    try:
        from google import genai
        gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
        print("✅ Gemini 클라이언트 준비 완료")
    except Exception as e:
        print(f"❌ Gemini 설정 실패: {e}")
        GEMINI_AVAILABLE = False
else:
    print("ℹ️ Gemini API 스킵 (EXAONE만 테스트)")

---
# 📊 테스트 데이터 정의

In [ ]:
# 채용공고 텍스트 (수정 가능)
JOB_POSTING_TEXT = """
카카오 백엔드 개발자 채용

■ 자격요건
- 경력 3년 이상
- Python, Java 중 1개 이상 능숙
- REST API 설계 및 개발 경험
- RDBMS 활용 경험 (MySQL, PostgreSQL)

■ 우대사항
- Kubernetes, Docker 경험
- 대용량 트래픽 처리 경험
- AWS, GCP 클라우드 경험

■ 주요업무
- 카카오 서비스 백엔드 API 개발
- 대규모 분산 시스템 설계 및 운영
"""

# 이력서 텍스트 (수정 가능)
RESUME_TEXT = """
홍길동 | 백엔드 개발자 | 경력 4년

■ 경력
1. ABC 스타트업 (2020-2022)
   - Python/FastAPI 기반 API 서버 개발
   - PostgreSQL 데이터베이스 설계

2. XYZ 테크 (2022-현재)
   - Spring Boot 마이크로서비스 개발
   - AWS EKS 기반 인프라 운영

■ 기술스택
- Python, Java, TypeScript
- FastAPI, Spring Boot
- PostgreSQL, Redis, MongoDB
- AWS, Docker, Kubernetes
"""

print("✅ 테스트 데이터 준비 완료")

In [ ]:
# 꼬리질문 테스트 시나리오
FOLLOWUP_TEST_CASES = [
    {
        "id": "tech_deep_dive_1",
        "category": "기술 심화",
        "original_question": "프로젝트에서 성능 최적화 경험을 말씀해주세요.",
        "candidate_answer": "Redis 캐싱을 도입해서 응답 시간을 70% 줄였습니다.",
        "star_analysis": {"situation": "incomplete", "task": "missing", "action": "incomplete", "result": "present"},
        "expected_followup_type": "technical_deep_dive",
        "expected_keywords": ["TTL", "캐시 전략", "만료", "무효화", "구체적"]
    },
    {
        "id": "star_complement_1",
        "category": "STAR 보완",
        "original_question": "가장 어려웠던 프로젝트 경험을 말씀해주세요.",
        "candidate_answer": "대용량 트래픽 처리 프로젝트에서 서비스가 다운되는 상황을 겪었습니다.",
        "star_analysis": {"situation": "present", "task": "missing", "action": "missing", "result": "missing"},
        "expected_followup_type": "star_complement",
        "expected_keywords": ["역할", "과제", "해결", "구체적으로", "담당"]
    },
    {
        "id": "tech_deep_dive_2",
        "category": "기술 심화",
        "original_question": "MSA 전환 경험에 대해 말씀해주세요.",
        "candidate_answer": "모놀리틱 아키텍처를 마이크로서비스로 분리했고, Kafka를 이용해 서비스 간 통신을 구현했습니다.",
        "star_analysis": {"situation": "incomplete", "task": "incomplete", "action": "present", "result": "missing"},
        "expected_followup_type": "technical_deep_dive",
        "expected_keywords": ["분리 기준", "도메인", "트랜잭션", "일관성", "어떻게"]
    },
    {
        "id": "verification_1",
        "category": "검증 질문",
        "original_question": "본인이 직접 설계한 시스템에 대해 설명해주세요.",
        "candidate_answer": "결제 시스템을 설계했는데 하루 100만 건을 처리할 수 있습니다.",
        "star_analysis": {"situation": "missing", "task": "incomplete", "action": "incomplete", "result": "present"},
        "expected_followup_type": "verification",
        "expected_keywords": ["직접", "어떤 부분", "팀", "역할", "설계"]
    },
    {
        "id": "star_complement_2",
        "category": "STAR 보완 (결과)",
        "original_question": "협업 과정에서 갈등을 해결한 경험이 있나요?",
        "candidate_answer": "기획팀과 일정 문제로 갈등이 있었는데, 주 2회 싱크업 미팅을 제안해서 진행했습니다.",
        "star_analysis": {"situation": "present", "task": "present", "action": "present", "result": "missing"},
        "expected_followup_type": "star_complement",
        "expected_keywords": ["결과", "개선", "이후", "효과", "변화"]
    }
]

print(f"📋 총 {len(FOLLOWUP_TEST_CASES)}개의 꼬리질문 테스트 케이스 준비 완료")

---
# 🛠️ 유틸리티 함수

In [ ]:
@dataclass
class LLMResult:
    """LLM 결과 저장"""
    model_name: str
    raw_output: str
    parsed_json: Optional[Dict[str, Any]]
    json_valid: bool
    response_time: float
    tokens_generated: int = 0
    tps: float = 0.0
    quality_scores: Dict[str, float] = field(default_factory=dict)
    error: Optional[str] = None

def extract_json_from_text(text: str) -> Optional[Dict[str, Any]]:
    """텍스트에서 JSON 추출"""
    try:
        return json.loads(text.strip())
    except:
        pass
    
    json_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if json_match:
        try:
            return json.loads(json_match.group(1).strip())
        except:
            pass
    
    brace_match = re.search(r'\{[\s\S]*\}', text)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except:
            pass
    
    return None

def evaluate_followup_quality(parsed: Dict, test_case: Dict) -> Dict[str, float]:
    """꼬리질문 품질 평가"""
    scores = {}
    
    # 1. JSON 구조 점수
    required_keys = ["followup_type", "question", "focus_area"]
    present_keys = sum(1 for k in required_keys if k in parsed)
    scores["structure"] = (present_keys / len(required_keys)) * 100
    
    # 2. 타입 일치 점수
    if parsed.get("followup_type") == test_case["expected_followup_type"]:
        scores["type_match"] = 100
    elif parsed.get("followup_type") in ["star_complement", "technical_deep_dive", "verification"]:
        scores["type_match"] = 50
    else:
        scores["type_match"] = 0
    
    # 3. 키워드 포함 점수
    question = parsed.get("question", "")
    expected_keywords = test_case.get("expected_keywords", [])
    keyword_hits = sum(1 for kw in expected_keywords if kw in question)
    scores["keyword_match"] = (keyword_hits / len(expected_keywords)) * 100 if expected_keywords else 0
    
    # 4. 질문 길이 적절성
    if 10 <= len(question) <= 100:
        scores["question_length"] = 100
    elif 5 <= len(question) <= 150:
        scores["question_length"] = 70
    else:
        scores["question_length"] = 30
    
    # 5. 한국어 비율
    korean_chars = len(re.findall(r'[가-힣]', question))
    scores["korean_ratio"] = min(100, (korean_chars / max(1, len(question))) * 200)
    
    # 종합 점수
    weights = {"structure": 0.2, "type_match": 0.25, "keyword_match": 0.25, "question_length": 0.15, "korean_ratio": 0.15}
    scores["total"] = sum(scores.get(k, 0) * w for k, w in weights.items())
    
    return scores

# 결과 저장
all_results: List[LLMResult] = []
print("✅ 유틸리티 함수 정의 완료")

---
# 🎯 프롬프트 정의

In [ ]:
# 종합 분석 프롬프트
ANALYSIS_PROMPT = """
당신은 채용 전문가입니다. 아래 이력서와 채용공고를 분석하여 JSON 형식으로 응답해주세요.

## 이력서
{resume_text}

## 채용공고
{job_posting_text}

## 출력 형식
{{
  "job_posting": {{"company": "회사명", "position": "포지션", "skills": {{"required": [], "preferred": []}}}},
  "resume": {{"total_experience": "경력", "skills": [], "strengths": [], "weaknesses": []}},
  "matching": {{"overall_grade": "A-F", "overall_score": 0, "skill_matching": {{"matched": [], "missing": []}}}}
}}

JSON만 출력하세요.
"""

# 꼬리질문 생성 프롬프트
FOLLOWUP_PROMPT = """
당신은 전문 면접관입니다. 지원자의 답변을 분석하고 적절한 꼬리질문을 생성하세요.

## 면접 질문
{original_question}

## 지원자 답변
{candidate_answer}

## STAR 분석
- Situation: {star_situation}
- Task: {star_task}
- Action: {star_action}
- Result: {star_result}

## 규칙
1. 부족한 STAR 요소 보완
2. 기술적 세부사항 확인
3. 한국어로 자연스러운 질문

## 출력 형식
{{"followup_type": "star_complement|technical_deep_dive|verification", "question": "질문", "focus_area": "목적", "expected_answer_elements": []}}

JSON만 출력하세요.
"""

def create_followup_prompt(test_case: Dict) -> str:
    star = test_case["star_analysis"]
    return FOLLOWUP_PROMPT.format(
        original_question=test_case["original_question"],
        candidate_answer=test_case["candidate_answer"],
        star_situation=star["situation"],
        star_task=star["task"],
        star_action=star["action"],
        star_result=star["result"]
    )

print("✅ 프롬프트 정의 완료")

---
# 🚀 모델 테스트: EXAONE-3.5-7.8B (L4 GPU)

In [ ]:
MODEL_7B = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"

print(f"🔄 {MODEL_7B} 로딩 중...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

try:
    tokenizer_7b = AutoTokenizer.from_pretrained(MODEL_7B, trust_remote_code=True)
    model_7b = AutoModelForCausalLM.from_pretrained(
        MODEL_7B,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    print(f"✅ {MODEL_7B} 로딩 완료")
    print(f"📊 GPU 메모리: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
except Exception as e:
    print(f"❌ 모델 로딩 실패: {e}")
    model_7b = None

In [ ]:
# 7.8B 꼬리질문 테스트
def run_exaone_test(model, tokenizer, test_cases, model_name):
    results = []
    
    for i, case in enumerate(test_cases):
        print(f"\n📌 테스트 {i+1}/{len(test_cases)}: {case['id']}")
        prompt = create_followup_prompt(case)
        
        messages = [
            {"role": "system", "content": "당신은 면접관입니다. JSON 형식으로만 응답하세요."},
            {"role": "user", "content": prompt}
        ]
        
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        
        start_time = time.time()
        with torch.no_grad():
            output_ids = model.generate(
                input_ids, max_new_tokens=512, temperature=0.1, do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        response_time = time.time() - start_time
        
        new_tokens = output_ids.shape[1] - input_ids.shape[1]
        output_text = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)
        parsed = extract_json_from_text(output_text)
        
        quality_scores = evaluate_followup_quality(parsed, case) if parsed else {}
        tps = new_tokens / response_time if response_time > 0 else 0
        
        result = LLMResult(
            model_name=model_name, raw_output=output_text, parsed_json=parsed,
            json_valid=parsed is not None, response_time=response_time,
            tokens_generated=new_tokens, tps=tps, quality_scores=quality_scores
        )
        results.append(result)
        
        print(f"⏱️ {response_time:.2f}s | TPS: {tps:.1f} | Score: {quality_scores.get('total', 0):.1f}")
        if parsed:
            print(f"📝 질문: {parsed.get('question', 'N/A')[:80]}...")
    
    return results

if model_7b:
    print("\n" + "="*60)
    print("🚀 EXAONE-3.5-7.8B 꼬리질문 테스트 시작")
    print("="*60)
    results_7b = run_exaone_test(model_7b, tokenizer_7b, FOLLOWUP_TEST_CASES, "EXAONE-3.5-7.8B")
    all_results.extend(results_7b)
    
    del model_7b, tokenizer_7b
    torch.cuda.empty_cache()
    print("\n🧹 GPU 메모리 정리 완료")

---
# 🚀 모델 테스트: EXAONE-3.5-32B (A100 GPU)

In [ ]:
# GPU 메모리 확인 후 32B 로드
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"📊 현재 GPU 메모리: {gpu_memory_gb:.2f} GB")

if gpu_memory_gb >= 35:
    MODEL_32B = "LGAI-EXAONE/EXAONE-3.5-32B-Instruct"
    print(f"\n🔄 {MODEL_32B} 로딩 중...")
    
    try:
        tokenizer_32b = AutoTokenizer.from_pretrained(MODEL_32B, trust_remote_code=True)
        model_32b = AutoModelForCausalLM.from_pretrained(
            MODEL_32B,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16
        )
        print(f"✅ {MODEL_32B} 로딩 완료")
        print(f"📊 GPU 메모리: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    except Exception as e:
        print(f"❌ 로딩 실패: {e}")
        model_32b = None
else:
    print(f"⚠️ GPU 메모리 부족 ({gpu_memory_gb:.1f}GB < 35GB)")
    print("   → EXAONE 32B 테스트 스킵. A100 GPU 필요.")
    model_32b = None

In [ ]:
if model_32b:
    print("\n" + "="*60)
    print("🚀 EXAONE-3.5-32B 꼬리질문 테스트 시작")
    print("="*60)
    
    results_32b = run_exaone_test(model_32b, tokenizer_32b, FOLLOWUP_TEST_CASES, "EXAONE-3.5-32B")
    all_results.extend(results_32b)
    
    del model_32b, tokenizer_32b
    torch.cuda.empty_cache()
    print("\n🧹 GPU 메모리 정리 완료")
else:
    print("ℹ️ 32B 테스트 스킵")

---
# 📊 결과 분석

In [ ]:
import pandas as pd

def summarize_results(results):
    data = []
    for r in results:
        data.append({
            "Model": r.model_name,
            "Test ID": r.test_case_id if hasattr(r, 'test_case_id') else 'N/A',
            "JSON Valid": "✅" if r.json_valid else "❌",
            "Time (s)": f"{r.response_time:.2f}",
            "TPS": f"{r.tps:.1f}",
            "Score": f"{r.quality_scores.get('total', 0):.1f}" if r.quality_scores else "N/A"
        })
    return pd.DataFrame(data)

if all_results:
    df = summarize_results(all_results)
    print("\n📊 테스트 결과 요약")
    print("="*60)
    display(df)
    
    # 모델별 평균
    print("\n📈 모델별 평균 성능")
    for model in df["Model"].unique():
        model_df = df[df["Model"] == model]
        avg_time = sum(float(t) for t in model_df["Time (s)"]) / len(model_df)
        avg_score = sum(float(s) for s in model_df["Score"] if s != "N/A") / len(model_df)
        json_success = sum(1 for v in model_df["JSON Valid"] if v == "✅") / len(model_df) * 100
        print(f"  {model}: 평균 {avg_time:.2f}s, 점수 {avg_score:.1f}, JSON 성공률 {json_success:.0f}%")
else:
    print("⚠️ 테스트 결과가 없습니다.")

## 5. 비용 분석

### GPU 서버 비용 (월간)
| 구성 | 시간당 | 월간 (24/7) |
|------|--------|------------|
| L4 × 1 (7.8B) | $0.80 | **$576** |
| A100 × 1 (32B) | $3.00 | **$2,160** |

### Gemini API 비용 (Fallback)
- 입력: $0.075/1M tokens
- 출력: $0.30/1M tokens
- 예상 Fallback 비용: ~$1-5/월 (10-20% 분기 시)

### 손익분기점
- L4 (7.8B): 월 ~15,000 요청 이상이면 Gemini API 대비 효율적

## 6. 분기점 임계값

| 조건 | 임계값 | 대응 |
|------|-------|------|
| vLLM 대기열 | ≥ 8 | Gemini 분기 |
| 응답 타임아웃 | > 10초 | Gemini 분기 |
| 헬스 체크 실패 | - | Gemini Fallback |

## 7. 서버리스 전략

### 스케일링 정책
| 지표 | 스케일 업 | 스케일 다운 |
|------|----------|------------|
| GPU 사용률 | > 80% (5분) | < 30% (15분) |
| 대기열 크기 | > 10 | < 2 |

### Cold Start 대응
- 최소 1대 Warm 인스턴스 유지
- Cold Start 동안 100% Gemini Fallback

## 8. 결론

### 최종 선정
| 항목 | 7.8B | **32B** ✅ |
|------|------|------------|
| 꼬리질문 품질 | 75-80점 | **85-90점** |
| 응답 시간 | ~10-15초 | ~20-30초 |
| 권장 용도 | 일반 채팅 | **면접 꼬리질문** |
| GPU | L4 24GB | A100 40GB |
| 월 비용 | ~$576 | ~$2,160 |

> ✅ **결론**: 면접 모드(꼬리질문 생성)에는 **EXAONE-3.5-32B** 권장

In [ ]:
# 결과 저장
if all_results:
    results_dict = {
        "test_date": time.strftime("%Y-%m-%d %H:%M:%S"),
        "results": [
            {
                "model": r.model_name,
                "test_id": getattr(r, 'test_case_id', 'N/A'),
                "json_valid": r.json_valid,
                "response_time": r.response_time,
                "tps": r.tps,
                "quality_scores": r.quality_scores
            }
            for r in all_results
        ]
    }
    
    with open("benchmark_results.json", "w", encoding="utf-8") as f:
        json.dump(results_dict, f, ensure_ascii=False, indent=2)
    
    print("✅ 결과가 benchmark_results.json에 저장되었습니다.")